# Import Library

In [2]:
!pip install Sastrawi
!pip install wordcloud
!pip install nltk
!pip install gensim -U

In [3]:
''' jalankan ini jika ada eror 'numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject'
. Jika berhasil dan dijalankan 2x terjadi eror, jadikan comment'''

#!pip install -qq numpy==1.26.4 gensim
#get_ipython().kernel.do_shutdown(restart=True)

" jalankan ini jika ada eror 'numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject'\n. Jika berhasil dan dijalankan 2x terjadi eror, jadikan comment"

In [4]:
import pandas as pd
pd.options.mode.chained_assignment = None

import datetime as dt
import re
import string
import csv
import requests
from io import StringIO
import tensorflow as tf

import numpy as np
seed=0
np.random.seed(seed)
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error
from sklearn.model_selection import cross_val_score, learning_curve, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB


import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.test.utils import common_texts
from gensim.models import Word2Vec
from tensorflow import keras
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from wordcloud import WordCloud


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Olah Dataset

In [5]:
!pip install google-play-scraper
from google_play_scraper import app, reviews, Sort, reviews_all
import pandas as pd
pd.options.mode.chained_assignment = None

In [6]:
count = 30000
scrap_review = reviews_all(
    'id.co.bri.brimo',
    lang='id',
    country='id',
    sort=Sort.MOST_RELEVANT,
    count=count
)

In [7]:
scrap_review = scrap_review[:count]

In [8]:
app_reviews_df = pd.DataFrame(scrap_review)
jumlah_ulasan, jumlah_kolom = app_reviews_df.shape
print('Banyak ulasan:', jumlah_ulasan)
print('Banyak kolom:', jumlah_kolom)

Banyak ulasan: 30000
Banyak kolom: 11


In [9]:
app_reviews_df.shape

(30000, 11)

In [10]:
clean_df = app_reviews_df.dropna()
clean_df = clean_df.drop_duplicates()
clean_df.shape

(26496, 11)

In [11]:
clean_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,24868505-ec59-4b9f-9814-652642d8cc5b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Hasil Update hanya mengutamakan tampilan saja ...,1,7,2.1.0,2021-01-13 22:05:02,"Hai, Sobat BRI. Mohon maaf atas kendala yang d...",2025-01-12 11:10:06,2.1.0
1,1310ede7-796f-40f8-8965-a5cfd49ab556,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,setelah update terakhir selalu muncul notifika...,2,24,2.81.0,2025-04-02 01:32:21,"Hai Sobat BRI, mohon maaf atas ketidaknyamanan...",2025-04-02 02:31:59,2.81.0
2,39c2205a-65d9-4871-b08f-e72eea8fa526,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,buka aplikasi Brimo tiba2 mental keluar sendir...,4,40,2.81.0,2025-03-25 02:58:54,"Hai Sobat BRI, mohon maaf atas ketidaknyamanan...",2025-03-25 03:28:17,2.81.0
3,6af367cf-4fce-4f7d-a818-90140d3337ce,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,awalnya ngebug gitu tiap mau tf ke ewallet pas...,1,3,2.80.0,2025-04-10 01:53:53,"Hai Sobat BRI, mohon maaf atas kendala login B...",2025-04-10 02:13:24,2.80.0
4,ee7e5f6f-2265-4d7d-967b-2419ab4edd03,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Buat yg keganggu dikit2 update tak kasi tau, u...",5,2141,2.79.0,2025-03-13 14:10:57,Terima kasih atas ulasannya. Semoga aplikasi B...,2025-03-14 01:10:42,2.79.0


In [12]:
clean_df = clean_df[['content', 'score', 'thumbsUpCount']]
clean_df.head()

,content,score,thumbsUpCount
0,Hasil Update hanya mengutamakan tampilan saja ...,1,7
1,setelah update terakhir selalu muncul notifika...,2,24
2,buka aplikasi Brimo tiba2 mental keluar sendir...,4,40
3,awalnya ngebug gitu tiap mau tf ke ewallet pas...,1,3
4,"Buat yg keganggu dikit2 update tak kasi tau, u...",5,2141


In [13]:
clean_df.shape

(26496, 3)

In [14]:
clean_df.to_csv('brimo.csv', index=False)